# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rifkiay/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule (plain words):** flag a page as a review priority if it has meaningful search visibility but is underperforming — either its CTR is well below what its position tier normally gets (Fix CTR pattern), or it sits at striking distance (position 11-20) with real impression volume already present (Quick Win pattern).

**Reason codes this rule can output:**
- `low_ctr_for_position`: CTR is at least 30% below the tier's expected weighted CTR
- `striking_distance_opportunity`: position 11-20 with impressions at or above the tier's median (20, from Signal 2)
- `no_action_needed`: neither condition triggers

**Action label:** `review_ctr_and_snippet`, `push_to_page_one`, or `monitor`, depending on which reason code fires.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"

In [2]:
# Signal 1: CTR vs Position tier — flag-linked to "Fix CTR"
signal1 = con.sql(f"""
WITH tiered AS (
  SELECT
    CASE
      WHEN gsc_avg_position <= 0 THEN 'no_data'
      WHEN gsc_avg_position <= 3 THEN 'top_3'
      WHEN gsc_avg_position <= 10 THEN 'page_1'
      WHEN gsc_avg_position <= 20 THEN 'striking'
      WHEN gsc_avg_position <= 50 THEN 'page_3_5'
      ELSE 'deep'
    END AS position_tier,
    gsc_clicks,
    gsc_impressions
  FROM read_parquet('{month_path}')
  WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
)
SELECT
  position_tier,
  COUNT(*) AS n,
  SUM(gsc_clicks) AS total_clicks,
  SUM(gsc_impressions) AS total_impressions,
  ROUND(SUM(gsc_clicks) * 100.0 / SUM(gsc_impressions), 3) AS weighted_ctr_pct
FROM tiered
GROUP BY position_tier
ORDER BY 
  CASE position_tier
    WHEN 'top_3' THEN 1 WHEN 'page_1' THEN 2 WHEN 'striking' THEN 3
    WHEN 'page_3_5' THEN 4 WHEN 'deep' THEN 5 ELSE 6 END
""").df()
signal1

,position_tier,n,total_clicks,total_impressions,weighted_ctr_pct
0,top_3,564173,204283.0,53559848.0,0.381
1,page_1,1456122,445828.0,137830113.0,0.323
2,striking,519223,92449.0,29386006.0,0.315
3,page_3_5,631491,76589.0,55944412.0,0.137
4,deep,276863,1509.0,3468464.0,0.044
5,no_data,163189,1174.0,468746.0,0.250


**Signal 1 — CTR vs Position tier:** CONFIRMED. Weighted CTR drops steadily as position gets worse: top_3 (0.381%) → page_1 (0.323%) → striking (0.315%) → page_3_5 (0.137%) → deep (0.044%). This supports FlyRank's "Fix CTR" flag assumption — a page with a low CTR relative to its position tier is a real signal worth flagging.

In [3]:
signal2 = con.sql(f"""
SELECT
  CASE 
    WHEN gsc_avg_position > 10 AND gsc_avg_position <= 20 THEN 'striking_distance'
    ELSE 'other'
  END AS zone,
  COUNT(*) AS n,
  AVG(gsc_impressions) AS avg_impressions,
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY gsc_impressions) AS median_impressions
FROM read_parquet('{month_path}')
WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0
GROUP BY zone
""").df()
signal2

,zone,n,avg_impressions,median_impressions
0,striking_distance,519223,56.596118,20.0
1,other,2928649,85.637725,17.0


**Signal 2 — Volume at striking distance:** MIXED. The median impressions for striking-distance pages (20) is actually higher than the "other" bucket's median (17), which supports real demand at this position. But the average comparison is misleading — "other" bagged in top_3/page_1 pages with huge impression outliers, pulling that average up. The median comparison is the fairer read here, and it still supports the "quick win" assumption: pages just outside page 1 carry meaningful, not negligible, demand.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = 0.5 × ctr_gap_score + 0.5 × striking_distance_score, computed per page from March 2026 data. Both components use only current-window signals (no future data, no product flags).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Aggregate daily rows to page-level (month of March 2026)
page_level = con.sql(f"""
SELECT
    content_hash_id,
    client_hash_id,
    AVG(gsc_avg_position) AS avg_position,
    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks
FROM read_parquet('{month_path}')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
GROUP BY content_hash_id, client_hash_id
HAVING SUM(gsc_impressions) > 0
""").df()

page_level["ctr"] = page_level["total_clicks"] / page_level["total_impressions"] * 100

# 2. Assign position tier (same buckets as Signal 1)
def tier_of(pos):
    if pos <= 0: return "no_data"
    elif pos <= 3: return "top_3"
    elif pos <= 10: return "page_1"
    elif pos <= 20: return "striking"
    elif pos <= 50: return "page_3_5"
    else: return "deep"

page_level["position_tier"] = page_level["avg_position"].apply(tier_of)

# 3. Expected CTR per tier, taken straight from Signal 1's weighted CTR table
expected_ctr = {
    "top_3": 0.381, "page_1": 0.323, "striking": 0.315,
    "page_3_5": 0.137, "deep": 0.044, "no_data": None
}
page_level["expected_ctr"] = page_level["position_tier"].map(expected_ctr)

# 4. CTR gap score: how far BELOW expected CTR this page sits (0 = meets/beats expectation, 1 = far below)
def ctr_gap(row):
    if row["expected_ctr"] is None or row["expected_ctr"] == 0:
        return 0
    gap = (row["expected_ctr"] - row["ctr"]) / row["expected_ctr"]
    return max(0, min(1, gap))  # clip to [0, 1]

page_level["ctr_gap_score"] = page_level.apply(ctr_gap, axis=1)

# 5. Striking distance score: 1 if in striking tier AND impressions >= tier median (20, from Signal 2)
page_level["striking_distance_score"] = (
    (page_level["position_tier"] == "striking") & (page_level["total_impressions"] >= 20)
).astype(int)

# 6. Final action score
page_level["action_score"] = (
    0.5 * page_level["ctr_gap_score"] + 0.5 * page_level["striking_distance_score"]
)

# 7. ONE reason code per page — pick whichever signal contributed more
def reason_code(row):
    if row["striking_distance_score"] == 1 and row["ctr_gap_score"] < 0.3:
        return "striking_distance_opportunity"
    elif row["ctr_gap_score"] >= 0.3:
        return "low_ctr_for_position"
    else:
        return "no_action_needed"

page_level["reason_code"] = page_level.apply(reason_code, axis=1)

action_map = {
    "low_ctr_for_position": "review_ctr_and_snippet",
    "striking_distance_opportunity": "push_to_page_one",
    "no_action_needed": "monitor"
}
page_level["action"] = page_level["reason_code"].map(action_map)

# 8. Rank (tie-break by impressions, so among equal scores the bigger opportunity ranks first) and write CSV
ranked = page_level.sort_values(
    ["action_score", "total_impressions"], ascending=[False, False]
).reset_index(drop=True)

from pathlib import Path
output_path = Path.cwd().parent / "outputs" / "baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
ranked.to_csv(output_path, index=False)

print("Rows written:", len(ranked))
print("Saved to:", output_path)
ranked.head(10)

Rows written: 176738
Saved to: d:\flayrank\flyrank-ml-internship-starter\work\outputs\baseline_action_score.csv


,content_hash_id,client_hash_id,avg_position,total_impressions,total_clicks,ctr,position_tier,expected_ctr,ctr_gap_score,striking_distance_score,action_score,reason_code,action
0,content_17d994b99d470434,client_62f4a7e64f5e0096,10.300517,19938.0,0.0,0.0,striking,0.315,1.0,1,1.0,low_ctr_for_position,review_ctr_and_snippet
1,content_cc9732f0da1d8d2d,client_fef1a8f436438636,13.318344,17115.0,0.0,0.0,striking,0.315,1.0,1,1.0,low_ctr_for_position,review_ctr_and_snippet
2,content_551a522809e0358c,client_fef1a8f436438636,16.918878,11355.0,0.0,0.0,striking,0.315,1.0,1,1.0,low_ctr_for_position,review_ctr_and_snippet
3,content_b154f6c2652cfeb9,client_73cda7b4e4f265ea,12.520681,11344.0,0.0,0.0,striking,0.315,1.0,1,1.0,low_ctr_for_position,review_ctr_and_snippet
4,content_155bbd621ff9af82,client_62f4a7e64f5e0096,11.592866,10245.0,0.0,0.0,striking,0.315,1.0,1,1.0,low_ctr_for_position,review_ctr_and_snippet
5,content_59a897a6ca5418df,client_20259bd6705d81d4,16.388320,8220.0,0.0,0.0,striking,0.315,1.0,1,1.0,low_ctr_for_position,review_ctr_and_snippet
6,content_b384057451493e31,client_23a62021009f63c4,16.261235,8188.0,0.0,0.0,striking,0.315,1.0,1,1.0,low_ctr_for_position,review_ctr_and_snippet
7,content_67dc193b282b586d,client_73cda7b4e4f265ea,15.890463,7535.0,0.0,0.0,striking,0.315,1.0,1,1.0,low_ctr_for_position,review_ctr_and_snippet
8,content_2631be5032e7adfc,client_23a62021009f63c4,19.177628,7179.0,0.0,0.0,striking,0.315,1.0,1,1.0,low_ctr_for_position,review_ctr_and_snippet
9,content_c48dc5490812302b,client_23a62021009f63c4,15.376083,6767.0,0.0,0.0,striking,0.315,1.0,1,1.0,low_ctr_for_position,review_ctr_and_snippet


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Reviewing the top 20 by action_score. A pattern worth flagging up front: CTR = 0.0 exactly (not just low) shows up repeatedly — this could mean genuinely zero clicks despite visibility, or a tracking/attribution gap for these specific pages. That uncertainty is called out per row below.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked.head(20).copy()

def why_here(row):
    if row["reason_code"] == "low_ctr_for_position":
        return (f"Position ~{row['avg_position']:.1f} ({row['position_tier']}) with "
                f"{row['total_impressions']:.0f} impressions but CTR of {row['ctr']:.2f}%, "
                f"vs {row['expected_ctr']:.3f}% expected for this tier.")
    else:
        return (f"Striking distance (pos ~{row['avg_position']:.1f}) with "
                f"{row['total_impressions']:.0f} impressions, above the tier's median volume.")

def confidence_note(row):
    if row["ctr"] == 0 and row["total_impressions"] >= 1000:
        return "LOW confidence until verified — a hard 0% CTR at this much volume is more likely a tracking gap than true zero engagement."
    elif row["ctr"] == 0:
        return "MEDIUM confidence — zero clicks is plausible at this volume, but still worth a manual spot-check."
    else:
        return "MEDIUM-HIGH confidence — CTR is measurably below tier expectation, not just zero by default."

def what_would_make_it_wrong(row):
    if row["ctr"] == 0:
        return "If this page's GSC click tracking is broken/misattributed, or the URL recently changed and impressions are still mapped to the old page."
    else:
        return "If this page is brand-new and hasn't accumulated enough clicks yet to reflect a stable CTR."

top20["why_here"] = top20.apply(why_here, axis=1)
top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(what_would_make_it_wrong, axis=1)

review_table = top20[["content_hash_id", "action", "reason_code", "why_here", "confidence_note", "what_would_make_it_wrong"]]
review_table

,content_hash_id,action,reason_code,why_here,confidence_note,what_would_make_it_wrong
0,content_17d994b99d470434,review_ctr_and_snippet,low_ctr_for_position,Position ~10.3 (striking) with 19938 impressio...,LOW confidence until verified — a hard 0% CTR ...,If this page's GSC click tracking is broken/mi...
1,content_cc9732f0da1d8d2d,review_ctr_and_snippet,low_ctr_for_position,Position ~13.3 (striking) with 17115 impressio...,LOW confidence until verified — a hard 0% CTR ...,If this page's GSC click tracking is broken/mi...
2,content_551a522809e0358c,review_ctr_and_snippet,low_ctr_for_position,Position ~16.9 (striking) with 11355 impressio...,LOW confidence until verified — a hard 0% CTR ...,If this page's GSC click tracking is broken/mi...
3,content_b154f6c2652cfeb9,review_ctr_and_snippet,low_ctr_for_position,Position ~12.5 (striking) with 11344 impressio...,LOW confidence until verified — a hard 0% CTR ...,If this page's GSC click tracking is broken/mi...
4,content_155bbd621ff9af82,review_ctr_and_snippet,low_ctr_for_position,Position ~11.6 (striking) with 10245 impressio...,LOW confidence until verified — a hard 0% CTR ...,If this page's GSC click tracking is broken/mi...
5,content_59a897a6ca5418df,review_ctr_and_snippet,low_ctr_for_position,Position ~16.4 (striking) with 8220 impression...,LOW confidence until verified — a hard 0% CTR ...,If this page's GSC click tracking is broken/mi...
6,content_b384057451493e31,review_ctr_and_snippet,low_ctr_for_position,Position ~16.3 (striking) with 8188 impression...,LOW confidence until verified — a hard 0% CTR ...,If this page's GSC click tracking is broken/mi...
7,content_67dc193b282b586d,review_ctr_and_snippet,low_ctr_for_position,Position ~15.9 (striking) with 7535 impression...,LOW confidence until verified — a hard 0% CTR ...,If this page's GSC click tracking is broken/mi...
8,content_2631be5032e7adfc,review_ctr_and_snippet,low_ctr_for_position,Position ~19.2 (striking) with 7179 impression...,LOW confidence until verified — a hard 0% CTR ...,If this page's GSC click tracking is broken/mi...
9,content_c48dc5490812302b,review_ctr_and_snippet,low_ctr_for_position,Position ~15.4 (striking) with 6767 impression...,LOW confidence until verified — a hard 0% CTR ...,If this page's GSC click tracking is broken/mi...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak pick pattern — lack of diversity in top 20:** all 20 top-ranked pages share the exact same reason code (`low_ctr_for_position`) and the exact same confidence flag (LOW). This isn't a coincidence — it's a flaw in the scoring formula. Because `ctr_gap_score` clips at 1.0 the moment CTR hits exactly 0%, every zero-CTR page ties at the maximum score, regardless of impression volume. This means `striking_distance_opportunity` pages never get a chance to surface in the top 20, even though Signal 2 showed real demand exists in that zone. A fix for a later iteration: use a continuous gap measure instead of clipping, or weight by impression volume more directly, so the two signals can actually compete for rank instead of one dominating by default.

**The CTR=0.0 pattern itself is the biggest risk in this queue:** a hard zero across pages with up to ~20K impressions is unusual enough that I would not trust these picks at face value — before recommending any of them for a rewrite, I'd verify the raw GSC export for a sample of these pages to rule out a tracking/attribution issue specific to this data slice.

**Leakage check:** confirmed no future-window or label-derived inputs were used. All features (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`) come from the same `month=2026-03` window being scored — nothing from April onward, and no FlyRank product flags (`health_score`, `priority_score`, `action_type`) were touched, since they aren't even present in this dataset.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
zero_ctr_count = (ranked["ctr"] == 0).sum()
total_count = len(ranked)
print(f"Pages with exactly 0% CTR: {zero_ctr_count:,} out of {total_count:,} ({zero_ctr_count/total_count*100:.1f}%)")

reason_code_counts = ranked["reason_code"].value_counts()
print("\nReason code distribution across ALL pages:")
print(reason_code_counts)

Pages with exactly 0% CTR: 107,901 out of 176,738 (61.1%)

Reason code distribution across ALL pages:
reason_code
low_ctr_for_position             131283
no_action_needed                  37230
striking_distance_opportunity      8225
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.